# Ollama — 내 컴퓨터 안에서 도는 모델

`1_2_gemini` 와 또 같은 코드입니다. 이번에는 **인터넷에 나가지 않습니다.**
모델이 내 PC 안에서 돕니다.

## 왜 로컬 모델을 쓰나

| | 클라우드 (OpenAI·Gemini) | 로컬 (Ollama) |
|---|---|---|
| 비용 | 토큰당 과금 | **공짜** |
| 데이터 | 외부 서버로 나간다 | **내 PC 밖으로 안 나간다** |
| 성능 | 좋다 | 내 PC 사양에 달렸다 |
| 품질 | 좋다 | **눈에 띄게 떨어진다** |
| 인터넷 | 필요 | 필요 없다 |

사내 기밀을 다루거나 비용이 부담될 때 씁니다.
대신 **품질을 양보해야 합니다.** 이 노트북 마지막에서 그 차이를 직접 보게 됩니다.

## 미리 해 둘 것

```bash
ollama serve            # 서버 띄우기
ollama pull llama3.2    # 모델 내려받기 (약 2GB)
```

## Step 1. 역시 `openai` 라이브러리

Ollama 전용 라이브러리가 따로 있지만, 여기서도 **`openai` 를 그대로** 씁니다.
Ollama 가 OpenAI 호환 입구를 제공하기 때문입니다.

In [7]:
import os
from dotenv import load_dotenv
from openai import OpenAI

## Step 2. `.env` 읽기

In [8]:
load_dotenv(override=True)

True

## Step 3. 주소를 보세요 — `localhost`

```
http://localhost:11434/v1
```

| 부분 | 뜻 |
|---|---|
| `http://` | `https` 가 아니다. 내 PC 안이라 암호화가 필요 없다 |
| `localhost` | **내 컴퓨터**. 인터넷으로 나가지 않는다 |
| `11434` | Ollama 가 쓰는 포트 번호 |
| `/v1` | OpenAI 호환 입구 |

> 여기서 연결이 안 되면 `ollama serve` 가 떠 있지 않은 것입니다.
> 터미널에서 `ollama list` 로 확인해 보세요.

In [9]:
ollama_url = os.getenv('OLLAMA_URL')
print(ollama_url)

## Step 4. API 키가 가짜여도 된다

```python
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
```

`api_key="ollama"` — 아무 글자나 넣었습니다. **내 PC 가 상대라 인증할 이유가 없기 때문입니다.**

그런데도 왜 넣을까요? `openai` 라이브러리가 **키 없이는 실행을 거부**하기 때문입니다.
그래서 형식만 맞춰 아무 값이나 넣어 줍니다.

> 로컬 모델에 요금이 붙지 않는 이유가 여기 있습니다.
> 과금할 계정 자체가 없습니다.

In [10]:
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

## Step 5. `system` 프롬프트

`1_1`·`1_2` 와 똑같습니다. 조건을 같게 두어야 **모델 차이만** 드러납니다.

In [11]:
system_prompt = """
당신은 IT 보안 전문가입니다. 
시스템 보안에 대한 전문 지식을 제공하며 새롭게 발생하는 보안 문제에 대해서도 대처 방법을 제공합니다
"""

## Step 6. `user` 프롬프트

In [12]:
user_prompt = """
나는 IT 회사의 보안 책임자입니다.
현재 가장 대두되고 있는 보안 이슈는 무엇이 있나요?
"""

## Step 7. `messages` 조립

이것도 완전히 같습니다. 여기까지 세 노트북에서 바뀐 것은 클라이언트 설정뿐입니다.

In [13]:
message = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

## Step 8. 호출 — `llama3.2`

```python
response = ollama.chat.completions.create(model="llama3.2", messages=message)
```

`llama3.2` 는 메타가 공개한 모델입니다. 작은 편(약 3B)이라 노트북에서도 돕니다.

| 모델 | 크기 | 특징 |
|---|---|---|
| `llama3.2` | ~2GB | 가볍다. 한국어는 약하다 |
| `qwen2.5` | 다양 | 한국어·중국어가 비교적 낫다 |
| `gemma3` | 다양 | 구글이 공개한 모델 |

`ollama pull <모델명>` 으로 받아 둔 모델만 쓸 수 있습니다.

> 첫 호출은 모델을 메모리에 올리느라 느립니다. 두 번째부터 빨라집니다.

In [14]:
response = ollama.chat.completions.create(model="llama3.2", messages=message)

APIConnectionError: Connection error.

## Step 9. 응답 구조 — 역시 같은 모양

`id` 가 `chatcmpl-183` 입니다. OpenAI 형식을 그대로 흉내 냈습니다.
`usage` 도 들어 있어 토큰 수를 볼 수 있습니다. **다만 요금은 붙지 않습니다.**

In [ ]:
import json

print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))

{
  "id": "chatcmpl-183",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": " current 보안 이슈중 가장 중요한 ones로, \n\n1.  **Zero-Day Attacks(ZDAs)**: 지속적으로 새로운 지점에 대한 공격이 발생합니다. 이러한 공격에서는 보안서버나 Netz워크가 현재 사용 중인 사 circumventing에 의해 공격되며, 자체적인 보안 체제에에 약점이 생기지 않는지 확인해 볼 필요가 있습니다. \n2.  **Artificial Intelligence(AI) and Machine Learning(ML) Threats**: AI 및 ML technology의 발전과 함께, 인공 지적은 의식 및 학습을 사용하는 위협은 increasingly 발생하고 있습니다. AI/ML가 특정-pattern에 대한 attack를 수행하는 것을 방지하기 위해, 보안 책임자들은 인공 지적의 성장을 이해하고 대응해야 합니다.\n\n3.  **Cloud Security 이슈**: Cloud의Growing popularity와 함께 Cloud Security는 Increasingly 중요합니다..cloud environment에서 data를 bảo vệ하는 것은 challenges가 있습니다.\n\n4.  **Cross-Session Attack**: Cross-Session Attack은 session token or cookie를 Tamper하는 것을 방지하기 위한 보안策정에 따라 발생하는 위협입니다.\n\n5.  **Insider Threats**: Organization 내부의 сотруд원 또는 직원으로부터 originate 하는 위협으로, 가장 큰 위협 중 하나입니다.\n\n만약 보안 책임자는 이러한 위협에 대해 대처 방법을 사용하여 bảo안 보장을

## Step 10. 답을 읽어 보세요 — 여기가 진짜 교훈

출력을 그냥 넘기지 말고 **문장을 하나씩 읽어 보세요.**

```
current 보안 이슈중 가장 중요한 ones로, ...
보안서버나 Netz워크가 현재 사용 중인 사 circumventing에 의해 ...
AI/ML가 특정-pattern에 대한 attack를 ...
```

한국어와 영어가 뒤섞이고, 문장이 끊기고, 말이 안 되는 부분이 있습니다.
**실수가 아니라 작은 모델의 실제 한계입니다.**

### 왜 이런 일이 생기나

| 원인 | 설명 |
|---|---|
| 모델 크기 | `llama3.2` 는 약 3B. GPT-4 급은 수백 B 규모 |
| 학습 데이터 | 영어 위주로 학습되어 한국어 비중이 작다 |
| 토큰 효율 | 한국어는 토큰을 더 많이 쓴다 (`1_5_tokenizer` 참고) |

### 그래서 어떻게 하나

- **한국어가 중요하면** `qwen2.5` 처럼 한국어를 더 학습한 모델을 쓴다
- **더 큰 모델**을 쓴다 (단, 그만큼 메모리와 시간이 든다)
- **분류·추출처럼 단순한 작업**만 로컬에 맡기고, 글을 쓰는 일은 클라우드에 맡긴다

> 로컬 모델이 나쁘다는 뜻이 아닙니다.
> **공짜와 비공개를 얻는 대신 품질을 내준다**는 거래를 이해하고 쓰자는 것입니다.

In [ ]:
print(response.choices[0].message.content)

 current 보안 이슈중 가장 중요한 ones로, 

1.  **Zero-Day Attacks(ZDAs)**: 지속적으로 새로운 지점에 대한 공격이 발생합니다. 이러한 공격에서는 보안서버나 Netz워크가 현재 사용 중인 사 circumventing에 의해 공격되며, 자체적인 보안 체제에에 약점이 생기지 않는지 확인해 볼 필요가 있습니다. 
2.  **Artificial Intelligence(AI) and Machine Learning(ML) Threats**: AI 및 ML technology의 발전과 함께, 인공 지적은 의식 및 학습을 사용하는 위협은 increasingly 발생하고 있습니다. AI/ML가 특정-pattern에 대한 attack를 수행하는 것을 방지하기 위해, 보안 책임자들은 인공 지적의 성장을 이해하고 대응해야 합니다.

3.  **Cloud Security 이슈**: Cloud의Growing popularity와 함께 Cloud Security는 Increasingly 중요합니다..cloud environment에서 data를 bảo vệ하는 것은 challenges가 있습니다.

4.  **Cross-Session Attack**: Cross-Session Attack은 session token or cookie를 Tamper하는 것을 방지하기 위한 보안策정에 따라 발생하는 위협입니다.

5.  **Insider Threats**: Organization 내부의 сотруд원 또는 직원으로부터 originate 하는 위협으로, 가장 큰 위협 중 하나입니다.

만약 보안 책임자는 이러한 위협에 대해 대처 방법을 사용하여 bảo안 보장을 시도합니다.


---

## 정리

| | 내용 |
|---|---|
| **코드는 똑같다** | `base_url` 만 `localhost` 로 바꾸면 된다 |
| **키는 형식만** | 내 PC 가 상대라 인증이 없다 |
| **공짜·비공개** | 토큰 요금이 없고 데이터가 밖으로 안 나간다 |
| **품질은 양보** | 작은 모델의 한국어는 확실히 약하다 |

### 직접 해 보기

1. `ollama pull qwen2.5` 로 받아서 `model` 만 바꿔 보세요. 한국어가 나아지나요?
2. 질문을 영어로 바꿔 보세요. 답의 품질이 달라지나요?
3. 와이파이를 끄고 다시 실행해 보세요. **그래도 동작합니다.**

### 다음 노트북

| 노트북 | 내용 |
|---|---|
| `1_4_llms` | 지금까지 본 네 모델에 같은 질문을 던져 한 번에 비교 |